# Breast Cancer Detection SVM Classifier

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.decomposition import PCA
from mpl_toolkits.mplot3d import Axes3D

## Load Data

The preprocessed and standardized data is loaded from `data/processedData/`. These CSVs were generated by the data preprocessor notebook, which applied standardization ($z = \frac{x - \mu}{\sigma}$) and an 80/20 stratified split (455 training, 114 test samples).

In [ ]:
X_train = pd.read_csv("../data/processedData/X_train_scaled.csv", index_col=0).values
Y_train = pd.read_csv("../data/processedData/Y_train.csv", index_col=0).values.ravel()

X_test = pd.read_csv("../data/processedData/X_test_scaled.csv", index_col=0).values
Y_test = pd.read_csv("../data/processedData/Y_test.csv", index_col=0).values.ravel()

print(f"Training samples: {X_train.shape[0]}, Test samples: {X_test.shape[0]}, Features: {X_train.shape[1]}")

## Train SVM Model

A **Support Vector Machine** with an RBF (Radial Basis Function) kernel is trained on the standardized features. The RBF kernel maps data into a higher-dimensional space, making it well-suited for non-linearly separable datasets like this one.

$$K(x_i, x_j) = \exp\left(-\gamma \|x_i - x_j\|^2\right)$$

Default hyperparameters (`C=1`, `gamma='scale'`) are used as a strong baseline.

In [ ]:
model = SVC(kernel="rbf")
model.fit(X_train, Y_train)
print("Model trained.")

## Evaluate on Test Set

Predictions are made on the held-out test set. **Recall for the malignant class (1) is the most critical metric** — a false negative (missed cancer) is more dangerous than a false positive.

In [ ]:
predictions = model.predict(X_test)

print("Accuracy:", accuracy_score(Y_test, predictions))
print()
print(classification_report(Y_test, predictions, target_names=["Benign (0)", "Malignant (1)"]))

### Confusion Matrix

Rows = actual class, columns = predicted class. The bottom-left cell (actual Malignant, predicted Benign) is the **false negative** count — the most important number to minimize in cancer detection.

In [ ]:
cm = confusion_matrix(Y_test, predictions)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Benign (0)", "Malignant (1)"],
    yticklabels=["Benign (0)", "Malignant (1)"],
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("SVM Confusion Matrix")
plt.tight_layout()
plt.show()

## 2D PCA Decision Boundary

The 30-dimensional feature space is reduced to 2 principal components for visualization. A new SVM is trained on the 2D-projected training data, and its decision boundary is plotted over the projected test points. This gives an intuitive view of how the model separates the two classes in a reduced space.

In [ ]:
pca = PCA(n_components=2)
X_train_pca = pca.fit_transform(X_train)
X_test_pca = pca.transform(X_test)

model_2d = SVC(kernel="rbf")
model_2d.fit(X_train_pca, Y_train)

x_min, x_max = X_test_pca[:, 0].min() - 1, X_test_pca[:, 0].max() + 1
y_min, y_max = X_test_pca[:, 1].min() - 1, X_test_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 500),
                     np.linspace(y_min, y_max, 500))

Z = model_2d.predict(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

plt.figure(figsize=(10, 7))
plt.contourf(xx, yy, Z, cmap=plt.cm.coolwarm, alpha=0.8)
plt.scatter(X_test_pca[:, 0], X_test_pca[:, 1],
            c=Y_test, edgecolors="k", cmap=plt.cm.coolwarm)
plt.title("SVM Decision Boundary (PCA Projection, Scaled Data)")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.tight_layout()
plt.show()

## 3D PCA Scatter Plot

Three principal components are used to visualize the separability of the two classes in a 3D space. Each point is a test sample colored by its true label (blue = benign, red = malignant).

In [ ]:
pca_3d = PCA(n_components=3)
X_3d = pca_3d.fit_transform(X_test)

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")
ax.scatter(X_3d[:, 0], X_3d[:, 1], X_3d[:, 2], c=Y_test, cmap="coolwarm", s=60)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
plt.title("SVM — 3D PCA Scatter (Test Set)")
plt.tight_layout()
plt.show()